# Controlled, stage-by-stage relocation workflow

Run the pipeline **one stage at a time**, inspect each intermediate product, and **tune
parameters minimally** before advancing — instead of a black-box end-to-end run. Each stage is:

> **PARAMS** (edit the marked values) → **RUN** (call the stage) → **INSPECT** (table/summary) → **PLOT**

Tuning uses `config.tune(cfg, ...)` which returns a *modified copy* of the frozen config (the
original `cfg0` is never mutated): scalars are replaced, the dict fields (`xcorr`, `pick_window`,
`pick_bandpass`) are **merged**, and the nested blocks (`hyp_control`, `ph2dt`) take a dict of field
overrides. To revert a stage, just re-run its cell with `cfg = config.tune(cfg0, ...)`.

## ⚠ Re-run discipline (read once)

`rereference` (stage 6) rewrites the SAC origin headers **in place** to the HYPOINVERSE solution —
it is the prerequisite for `dt.cc`. Consequence:

- **If you re-tune PICKING after running rereference, re-run the whole unit
  `waveforms → picking → hypoinverse → ph2dt+dtct → rereference` in order.** Re-running `waveforms`
  first resets the SAC origins to the catalog time, so picking starts from a known reference.
- Re-running a stage in place otherwise is fine; `rereference` is idempotent for a fixed `ARC_VM`.

`xcorr` (stage 7) cross-correlates **every** event pair and is the heavy step — keep it cheap with a
coarse `slide_step` and/or a small `SUBSET`, and launch JupyterLab under `taskset -c <cpulist>` on a
shared box. Drop to `slide_step=0.001` (the baseline grid) only for the final product.

In [ ]:
import os, sys
# Assumes the notebook runs from pipeline/notebooks/ ; otherwise `export PYTHONPATH=<repo root>`.
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "..")))
import pandas as pd
import matplotlib.pyplot as plt
from obspy import read

from pipeline import config, viz
from pipeline.core import (stations, waveforms, picking, hypoinverse, hypodd,
                           rereference, xcorr, sumio, focal_mechanism)
from pipeline.regression import compare

CLUSTER = "gwangyang"                 # gwangyang | kimcheon | jangsung | gyeongju
cfg0 = config.load_cluster(CLUSTER)   # pristine baseline config (never mutated)
cfg  = cfg0                           # `cfg` is the live, possibly-tuned config
ARC_VM = "kim1983"                    # velmodel whose .arc/.sum feeds ph2dt/rereference/xcorr
print(f"{cfg.name}: region={cfg.region}  wf_source={cfg.wf_source}")
print(f"  src (read-only inputs+baseline): {cfg.src_root}")
print(f"  outputs ->                       {cfg.output_root}")

## Global knobs

`SUBSET` limits the heavy stages to a few events for fast iteration (the located `.sum` then drives
xcorr's pairing). Set `SUBSET = None` for the full product. `XCORR_CORES` caps the xcorr worker pool
(also pin the kernel with `taskset`).

In [ ]:
SUBSET = None                # e.g. ["20210720161418", "20210827220322"]; None = all events
DEVICE = "cpu"               # PhaseNet device; CPU matches the baseline (GPU can drift picks)
VELMODELS = ("kim1983", "kim2011")
XCORR_CORES = 10             # xcorr pool cap; launch: taskset -c 0-9 jupyter lab ...
print("SUBSET:", SUBSET, "| DEVICE:", DEVICE, "| VELMODELS:", VELMODELS, "| XCORR_CORES:", XCORR_CORES)

## 1. Stations
Master station table → within `radius_km` of the epicenter → keep those with Z-component data, one
sensor each (precedence HH > EL > HG). **PARAMS:** `radius_km`, `sensor_priority`.

In [ ]:
cfg = config.tune(cfg0, radius_km=100.0)            # <-- PARAMS (edit & re-run)
used = stations.run_stations(cfg)
print(f"[stations] {len(used)} used stations")
display(used["Sensor"].value_counts().rename("n_stations"))
display(used.head(12))

## 2. Waveforms
Gather the selected-sensor 3-component SAC per event and inject headers (event/station coords +
**catalog** origin). Re-running this resets the SAC origins to catalog time.

In [ ]:
wf = waveforms.run_waveforms(cfg, events=SUBSET)
print(f"[waveforms] origin-ref = CATALOG | gathered {sum(wf.values())} SAC / {len(wf)} events")
display(pd.Series(wf, name="n_files").sort_index())
empties = [e for e, n in wf.items() if n == 0]
if empties: print("WARNING: events with no waveforms:", empties)

In [ ]:
# 3-component sanity for one event/station (Z/N/E + P/S picks once picking has run)
ev0 = sorted(wf)[0]
viz.plot_3c(cfg, ev0); plt.show()

## 3. Picking — PhaseNet (main tuning surface)
Windowed PhaseNet → best P+S pair → writes `picks/*.csv` and the SAC `a`/`t0` headers.
**PARAMS:** `p_threshold`, `s_threshold`, `pick_bandpass`, `pick_window`, `sp_max_gap_s`.
⚠ If you re-tune this *after* stage 6, re-run waveforms→picking→hypoinverse→…→rereference (see banner).

In [ ]:
cfg = config.tune(cfg,                              # <-- PARAMS (edit & re-run)
                  p_threshold=0.2, s_threshold=0.2, sp_max_gap_s=15.0,
                  pick_bandpass=dict(freqmin=1.0, freqmax=40.0),   # merged into the dict
                  pick_window=dict(vp=5.9, vs=3.0, evdp=15.0))     # merged into the dict
pk = picking.run_picking(cfg, events=SUBSET, device=DEVICE)
print(f"[picking] origin-ref = CATALOG | {sum(pk.values())} picks / {len(pk)} events")
display(pd.Series(pk, name="n_picks").sort_index())

In [ ]:
viz.plot_picks(cfg, sorted(pk)[0]); plt.show()    # busiest station, red=P / blue=S

## 4. HYPOINVERSE — absolute location
Writes the station + phase files once, then locates per velocity model. **PARAMS:** `velmodels`,
`hyp_control` (e.g. `MIN`, `ZTR`, `DIS`, `RMS`).

In [ ]:
cfg = config.tune(cfg, hyp_control=dict(MIN=4))     # <-- PARAMS (nested replace)
sums = hypoinverse.run_hypoinverse(cfg, velmodels=VELMODELS)
for vm in VELMODELS:
    s = sumio.read_sum(config.sum_file(cfg, vm))
    print(f"[hypoinverse] {vm}: {len(s)} located | mean depth {s.depth.mean():.2f} km | mean rms {s.rms.mean():.3f} s")
display(sumio.read_sum(config.sum_file(cfg, ARC_VM)).head(12))

In [ ]:
viz.map_catalog(cfg, velmodel=ARC_VM, source="sum"); plt.show()
viz.depth_sections(cfg, velmodel=ARC_VM, source="sum"); plt.show()
viz.cumulative_events(cfg, velmodel=ARC_VM); plt.show()

## 5. HypoDD backbone — ph2dt + dt.ct
`.arc(ARC_VM)` → S-flag patch → ncsn2pha (+ East-longitude fix) → ph2dt → catalog (dt.ct) relocation.
**PARAMS:** `ph2dt` (`MAXSEP`, `MAXDIST`, `MINLNK`, ...), and `hypodd_dtct.isolv` (1=SVD, 2=LSQR).

In [ ]:
cfg = config.tune(cfg, ph2dt=dict(MAXDIST=200, MAXSEP=10, MINLNK=1))   # <-- PARAMS (nested replace)
hypodd.prep_ph2dt(cfg, velmodel=ARC_VM)
hypodd.run_ph2dt(cfg)
reloc_ct = hypodd.run_dtct(cfg)
ct = sumio.read_reloc(reloc_ct)
print(f"[dtct] {len(ct)} events relocated -> {reloc_ct}")
display(ct[["id","lat","lon","depth"]].head(12))

In [ ]:
viz.map_catalog(cfg, velmodel=ARC_VM, source="reloc"); plt.show()
viz.depth_sections(cfg, velmodel=ARC_VM, source="reloc"); plt.show()

## 6. Re-reference SAC origins → the `.sum`  (prerequisite for dt.cc)
Rewrites each event's SAC origin (`nz*`) + picks (`a`/`t0`) to the HYPOINVERSE `ARC_VM` solution, in
place under `runs/<cluster>/waveforms_100km/`. Idempotent for a fixed `ARC_VM`.
**⚠ After this cell, re-tuning picking requires re-running waveforms first (see banner).**

In [ ]:
n = rereference.rereference_origins(cfg, velmodel=ARC_VM)
print(f"[rereference] origin-ref = {ARC_VM}.sum | {n} events re-referenced")
# confirm one event's SAC origin now equals the .sum origin
sdf = sumio.read_sum(config.sum_file(cfg, ARC_VM))
eid = sorted(wf)[0]; cus = int(sdf.id.iloc[0])
import glob as _g
f = _g.glob(os.path.join(config.event_wf_dir(cfg, eid), f"{eid}.*Z.sac"))[0]
tr = read(f)[0]; ot = tr.stats.starttime - tr.stats.sac.b
print(f"  event {eid}: SAC origin {ot}  vs  .sum {sdf.time.iloc[0]}")

## 7. Cross-correlation `dt.cc`  (HEAVY)
Slide-loop cross-correlation of **all** event pairs (P on Z; S on N/E, higher-cc wins). **PARAMS:**
`xcorr` dict — `cc_threshold`, `bandpass`, `pre`/`post`, **`slide_step`** (speed/precision knob;
0.001 = baseline grid, 0.01 = ~10× faster look). `cores` caps the pool.
**Pin the kernel:** `taskset -c 0-9 jupyter lab …`. Reduce cost with a small `SUBSET` (stages 2–6).

In [ ]:
cfg = config.tune(cfg, xcorr=dict(slide_step=0.01))   # <-- PARAMS (coarse=fast; 0.001=final)
xc = xcorr.run_xcorr(cfg, velmodel=ARC_VM, cores=XCORR_CORES)
print(f"[xcorr] {xc['pairs']} pairs x {xc['stations']} stations -> {xc['combined']}")

In [ ]:
viz.cc_histogram(cfg); plt.show()                  # cc-coefficient distribution (P vs S)

## 8. HypoDD `dt.cc` relocation

Relocate using the cross-correlation differential times — the high-end product. **PARAMS:** `DTCC_VARIANT`
(`default` | `no_main` | `kim2011`, per the cluster config).

**Automatic LSQR conditioning (a real step, not magic).** When the cluster is large enough to force LSQR
(`isolv=2`, the SVD `MAXDATA0` case), `run_dtcc` flexibly modulates two levers so the inversion is
well-conditioned and stable: (1) the inter-event **distance cutoffs** are scaled to the dt.ct cluster size,
so spatially peripheral / uncorrelated events drop out instead of taking wild steps; (2) the per-set
**DAMP** is auto-tuned until the condition number sits in HypoDD's recommended **~40–80** band. It is
deterministic — re-running reproduces the same parameters and the same relocation. The next cell prints the
calibration the run chose (`damping_calibration.txt`).

In [ ]:
DTCC_VARIANT = "default"                            # <-- PARAMS
reloc_cc = hypodd.run_dtcc(cfg, variant=DTCC_VARIANT)
cc = sumio.read_reloc(reloc_cc)
print(f"[dtcc:{DTCC_VARIANT}] {len(cc)} events relocated -> {reloc_cc}")
display(cc[["id","lat","lon","depth"]].head(12))

In [ ]:
# inspect the adaptive LSQR calibration this dt.cc run chose (LSQR clusters only; reproducible)
_cal = os.path.join(config.dtcc_dir(cfg), "damping_calibration.txt")
print(open(_cal).read() if os.path.exists(_cal)
      else "No adaptive calibration (SVD dt.cc) — damping/distance used as configured.")

In [ ]:
viz.compare_epicenters(cfg, velmodel=ARC_VM, variant=DTCC_VARIANT); plt.show()

## 9. Regression vs the frozen baseline
PASS/FAIL per stage (report-only for dt.cc). **Meaningful only for the full run** (`SUBSET = None`);
on a subset the event counts won't match the baseline.

In [ ]:
df = compare.compare_all(cfg, velmodels=VELMODELS)
with pd.option_context("display.width", 220, "display.max_columns", 60):
    display(df)
hard = df[df.passed.notna()]
print(f"{int(hard.passed.sum())}/{len(hard)} hard-gated stages PASS (dt.cc content is report-only)")

## 10. Focal mechanisms — phasenet_plus + SKHASH  (opt-in)

Double-couple mechanisms from **first-motion polarity + S/P amplitude ratio**, which only the
`phasenet_plus` picker emits. This runs an independent phasenet_plus sub-run (a separate
`output_root`, so the `stead` flow above is untouched), then SKHASH — which ray-traces takeoff
angles from the cluster velocity model and computes azimuths from the station geometry. Only
**quality A/B** ("fairly high confidence") solutions are kept. (Set `SUBSET` above to a few events
for a quick look; the sub-run re-gathers waveforms + re-picks with PhaseNet+.)

In [ ]:
from pipeline.core import pipeline as _pl                 # local alias (don't shadow the package)
cfg_fm = config.tune(cfg0, picker_weights="phasenet_plus",  # <-- PARAMS
                     output_root=os.path.join(config.RUNS_ROOT, f"{CLUSTER}_pnplus"),
                     fm_velmodel=ARC_VM, fm_quality_keep=("A", "B"), fm_use_sp_ratio=True)
_pl.run_cluster(cfg_fm, stage_from="stations", through="hypoinverse",
                velmodels=(ARC_VM,), device=DEVICE, events=SUBSET)   # phasenet_plus picks + location
fm = focal_mechanism.run_focal_mechanism(cfg_fm, velmodel=ARC_VM)    # SKHASH inversion
mech = pd.read_csv(config.fm_mech_csv(cfg_fm, ARC_VM))
display(mech[["event_id", "quality", "strike", "dip", "rake", "fault_plane_uncertainty",
              "num_p_pol", "num_sp_ratios", "azimuthal_gap", "sta_distribution_ratio"]])
hi = mech[mech.quality.isin(cfg_fm.fm_quality_keep)]
print(f"high-confidence [{'/'.join(cfg_fm.fm_quality_keep)}]: {len(hi)} / {len(mech)} mechanisms")

In [ ]:
# beachball for the best-constrained event (SKHASH writes one PNG per event, named by cuspid)
from matplotlib import image as mpimg
row = (hi.sort_values("quality").iloc[0] if len(hi) else mech.iloc[0])
png = os.path.join(config.fm_out_dir(cfg_fm, ARC_VM), f"{int(row.cuspid)}.png")
if os.path.exists(png):
    plt.figure(figsize=(5, 5)); plt.imshow(mpimg.imread(png)); plt.axis("off")
    plt.title(f"{row.event_id}  {row.quality}  s/d/r = "
              f"{row.strike:.0f}/{row.dip:.0f}/{row.rake:.0f}"); plt.show()
else:
    print("no beachball PNG for", row.event_id)

### Where to go next
- Tighten `dt.cc`: set `slide_step=0.001` in stage 7 and re-run stages 7→8.
- Parameter study: change one PARAMS value, re-run that stage + downstream, compare maps/sections.
- Add a new cluster: see the top-level `README.md` ("Adding a cluster") — write `clusters/<name>.py`
  and register it in `config.CLUSTER_NAMES` / `config.CLUSTER_SRC_DIRS`.
- Auxiliary QC (orientation, ZRT, waveform similarity): `notebooks/01_qc.ipynb`.